# IEEE-CIS - OHE-Projected CAPGD

Cross-dataset generalization of the LCLD g1+M1 result, per
`docs/constraint_evaluation_guidance.md` section 5 (Phase 2 re-scoped).
The cross-dataset feasibility audit (`docs/cross_dataset_feasibility_findings.md`)
established that the binding constraints on IEEE-CIS are OHE-validity
checks on ProductCD / card4 / card6 (adv pass 0.018 / 0.048 / 0.206),
not C/D non-negativity as originally hypothesised.

This notebook tests whether the constraint-aware-attack-recovers-attack-power
pattern from LCLD generalises:

1. **Unconstrained** - stock `capgd_attack` (matches the cross-dataset audit).
2. **OHE-projected** - identical inner loop, but after each PGD step we
   snap the three OHE blocks (ProductCD, card4, card6) to a valid one-hot
   via per-block argmax.

Expected outcome: filtered-success rate jumps from ~0.014% to ~50% with
the flipped-prediction count preserved, mirroring LCLD's 0.05% -> 50% jump
under g1-projection.

**Produces**
- `results/adv_examples/ieee_ohe_projection/ieee_neural_{unconstrained,oheproj}_seed{s}.parquet`
- `ieee_ohe_projection_results.csv` - per-(seed, attack) metrics + feasibility + filtered success rate
- `ieee_ohe_projection_summary.csv` - mean +/- std across seeds
- `ieee_ohe_projection_bars.png` - side-by-side comparison bar chart

**Scope.** IEEE-CIS only. 3 seeds, eps=0.1, sample_frac=0.1, identical model
hyperparameters to the cross-dataset and LCLD g1-projection runs so the
constrained numbers are directly comparable.

**Bootstrap cells 1-5** mirror the g1-projection notebook verbatim - Drive
mount, repo clone, deps install (requires runtime restart), dataset symlinks.


In [ ]:
# Cell 1: Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "results/adv_examples"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
print("Google Drive mounted.")


In [ ]:
# Cell 3: Clone or update repo
import os, shutil

REPO_URL = "https://github.com/iHaydenzZ/Capstone_FraudBench.git"
REPO_DIR = "/content/Capstone_FraudBench"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    os.chdir(REPO_DIR)
    !git pull
else:
    os.chdir("/content")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")
!git log --oneline -3


In [ ]:
# Cell 4: Install dependencies
!pip install "numpy<2.1" "scipy>=1.14,<1.15" "scikit-learn>=1.5" -q 2>&1 | tail -5
!pip install -e . --no-deps -q 2>&1 | tail -5
!pip install "numba>=0.61" -q 2>&1 | tail -3
!pip install xgboost torch art pyyaml joblib pandas matplotlib -q 2>&1 | tail -3

# --- IMPORTANT ---
# After this cell finishes, restart the runtime:
#   Runtime > Restart session  (or Ctrl+M then .)
# Then skip this cell and continue from Cell 5.
print("\n>>> RESTART THE RUNTIME NOW, then skip this cell and run from Cell 5. <<<")


In [ ]:
# Cell 5: Symlink datasets from Google Drive
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/Capstone_FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f"  Linked: {dataset_dir}/")
    else:
        print(f"  NOT FOUND: {dataset_dir}/ -- upload to {src}")

print("Dataset symlinks ready.")


In [ ]:
# Cell 6: Experiment configuration
import os
import numpy as np
import pandas as pd

DATASET = "ieee_cis"
SEEDS = [42, 123, 456]
EPSILON = 0.1
SAMPLE_FRAC = 0.1
ATTACK_PARAMS = {"epsilon": EPSILON, "steps": 10, "step_size": EPSILON / 4}
MODEL_PARAMS = {"epochs": 20, "hidden_dim": 128, "batch_size": 256, "lr": 0.001}

# OHE block tolerance for the validity check: |sum-1|<tol and |max-1|<tol.
TOL_OHE = 0.01

# Three OHE-validity blocks identified as binding by Phase 1 Cell 14
# (cross_dataset_feasibility.ipynb): ProductCD (0.018), card4 (0.048),
# card6 (0.206). All three are projected per-step.
OHE_PREFIXES = ["ProductCD", "card4", "card6"]

OUT_DIR = "results/adv_examples/ieee_ohe_projection"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output dir: {OUT_DIR}")
print(f"Plan: {DATASET} x {len(SEEDS)} seeds x 2 attacks = {len(SEEDS)*2} runs")


In [ ]:
# Cell 7: IEEE-CIS constraint checkers (copied from cross_dataset_feasibility.ipynb Cell 10)
import re
from typing import Any, Dict

def _to_float(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^\d.\-]", "", regex=True),
        errors="coerce",
    )

def get_scaler_and_num_names(preprocessor):
    for name, transformer, columns in preprocessor.pipeline.transformers_:
        if name == "num":
            return transformer.named_steps["scaler"], list(columns)
    raise RuntimeError("No 'num' branch on the preprocessor")

def inverse_transform_numeric(X_proc, num_feature_names, scaler):
    sanitize = lambda c: re.sub(r"[\[\]<>]", "_", c)
    sanitized_num = [sanitize(c) for c in num_feature_names]
    proc_cols = X_proc.columns.tolist()
    matched = [(raw, san) for raw, san in zip(num_feature_names, sanitized_num)
               if san in proc_cols]
    if not matched:
        return pd.DataFrame(index=X_proc.index)
    raw_names = [m[0] for m in matched]
    san_names = [m[1] for m in matched]
    idx_in_scaler = [num_feature_names.index(r) for r in raw_names]
    X_scaled = X_proc[san_names].values
    means = scaler.mean_[idx_in_scaler]
    scales = scaler.scale_[idx_in_scaler]
    return pd.DataFrame(X_scaled * scales + means, columns=raw_names, index=X_proc.index)

def check_ohe_valid(X_proc, raw_prefix, tol=TOL_OHE):
    cols = [c for c in X_proc.columns if c.startswith(raw_prefix + "_")]
    if not cols:
        return None
    ohe = X_proc[cols].values
    valid = (np.abs(ohe.sum(axis=1) - 1.0) < tol) & (np.abs(ohe.max(axis=1) - 1.0) < tol)
    return pd.Series(valid, index=X_proc.index)

C_PATTERN = re.compile(r"^C\d+$")
D_PATTERN = re.compile(r"^D\d+$")

def check_txn_amt_positive(df):
    if "TransactionAmt" not in df.columns:
        return None
    return _to_float(df["TransactionAmt"]) > 0

def check_c_nonneg(df):
    c_cols = [c for c in df.columns if C_PATTERN.match(c)]
    if not c_cols:
        return None
    vals = df[c_cols].apply(_to_float)
    return (vals >= -0.5).all(axis=1)

def check_d_nonneg(df):
    d_cols = [c for c in df.columns if D_PATTERN.match(c)]
    if not d_cols:
        return None
    vals = df[d_cols].apply(_to_float).fillna(0.0)
    return (vals >= -0.5).all(axis=1)

def _pass(s):
    return float(s.fillna(True).mean()) if s is not None else np.nan

def _bool_series(s, idx):
    if s is None:
        return pd.Series(True, index=idx)
    return s.fillna(True).astype(bool)

def ieee_feasibility(X_raw, X_proc):
    per = {
        "i_amt_positive":  _pass(check_txn_amt_positive(X_raw)),
        "i_c_nonneg":      _pass(check_c_nonneg(X_raw)),
        "i_d_nonneg":      _pass(check_d_nonneg(X_raw)),
        "i_product_ohe":   _pass(check_ohe_valid(X_proc, "ProductCD")),
        "i_card4_ohe":     _pass(check_ohe_valid(X_proc, "card4")),
        "i_card6_ohe":     _pass(check_ohe_valid(X_proc, "card6")),
    }
    idx = X_raw.index
    parts = [
        _bool_series(check_txn_amt_positive(X_raw),         idx),
        _bool_series(check_c_nonneg(X_raw),                 idx),
        _bool_series(check_d_nonneg(X_raw),                 idx),
        _bool_series(check_ohe_valid(X_proc, "ProductCD"),  idx),
        _bool_series(check_ohe_valid(X_proc, "card4"),      idx),
        _bool_series(check_ohe_valid(X_proc, "card6"),      idx),
    ]
    agg = parts[0]
    for p in parts[1:]:
        agg = agg & p
    return per, float(agg.mean())


In [ ]:
# Cell 8: Projection metadata - one OHE block per binding constraint.
# Mirrors build_term_proj_info() from g1_projection_attack.ipynb but
# generalised to any prefix.

def build_ohe_block(processed_feature_names, raw_prefix):
    """Locate the indices of all processed columns belonging to one OHE block."""
    indices = [i for i, c in enumerate(processed_feature_names)
               if c.startswith(raw_prefix + "_")]
    if not indices:
        return None
    return {"prefix": raw_prefix, "indices": indices}

def build_ieee_ohe_blocks(processed_feature_names, prefixes=OHE_PREFIXES):
    blocks = []
    for prefix in prefixes:
        block = build_ohe_block(processed_feature_names, prefix)
        if block is None:
            print(f"  WARN: no OHE columns found for prefix {prefix!r}; skipping")
            continue
        blocks.append(block)
    if not blocks:
        raise RuntimeError(f"No OHE blocks found for any of {prefixes}")
    return blocks


In [ ]:
# Cell 9: OHE-projected CAPGD - copy of attacks.capgd.capgd_attack with
# per-step argmax-snap on each OHE block. No formula projection (no
# installment-style derived feature on IEEE-CIS) and no mutability mask
# (deferred to a follow-up cell once MVP results are validated).
import torch
import torch.nn as nn
from typing import Any, Dict, List
from attacks.capgd import project_constraints

def project_ohe_block_tensor(x_adv, block):
    """In-place argmax-snap of one OHE block on x_adv. Caller is responsible
    for being inside torch.no_grad()."""
    idx = block["indices"]
    ohe = x_adv[:, idx]
    argmax = ohe.argmax(dim=1)
    snapped = torch.zeros_like(ohe)
    snapped[torch.arange(ohe.size(0), device=x_adv.device), argmax] = 1.0
    x_adv[:, idx] = snapped
    return x_adv

def project_all_ohe_blocks(x_adv, blocks):
    for block in blocks:
        x_adv = project_ohe_block_tensor(x_adv, block)
    return x_adv

def capgd_attack_ohe_projected(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    schema,
    feature_types: Dict[str, str],
    ohe_blocks: List[dict],
    params: Dict[str, Any] = None,
) -> pd.DataFrame:
    if params is None:
        params = {}
    epsilon = params.get("epsilon", 0.1)
    steps = params.get("steps", 10)
    step_size = params.get("step_size", epsilon / 4)

    if not hasattr(model, "model") or not isinstance(model.model, nn.Module):
        print("Warning: non-PyTorch model; returning clean data.")
        return X

    torch_model = model.model
    device = model.device
    torch_model.train(False)

    X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device)
    feature_names = X.columns.tolist()

    # Initial random perturbation, then schema clip, then OHE snap.
    noise = torch.zeros_like(X_tensor).uniform_(-epsilon, epsilon)
    x_adv = X_tensor + noise
    x_adv = project_constraints(x_adv, X_tensor, schema, feature_names, feature_types)
    with torch.no_grad():
        x_adv = project_all_ohe_blocks(x_adv, ohe_blocks)
    x_adv = x_adv.detach()
    x_adv.requires_grad = True

    use_logits = hasattr(model, "_use_logits") and model._use_logits
    criterion = nn.BCEWithLogitsLoss() if use_logits else nn.BCELoss()

    for _ in range(steps):
        outputs = torch_model(x_adv)
        loss = criterion(outputs, y_tensor)
        torch_model.zero_grad()
        loss.backward()

        with torch.no_grad():
            x_adv = x_adv + step_size * x_adv.grad.sign()
            if epsilon > 0:
                delta = torch.clamp(x_adv - X_tensor, -epsilon, epsilon)
                x_adv = X_tensor + delta
            x_adv = project_constraints(x_adv, X_tensor, schema, feature_names, feature_types)
            x_adv = project_all_ohe_blocks(x_adv, ohe_blocks)
            x_adv.requires_grad = True

    return pd.DataFrame(x_adv.detach().cpu().numpy(), columns=feature_names, index=X.index)


In [ ]:
# Cell 10: Main loop - train one MLP per seed, run both attacks on the same model.
import time
from datasets.loader import load_dataset
from datasets.splitter import split_dataset
from preprocessing.processor import DataPreprocessor
from constraints.schema import ConstraintSchema
from models.neural import NeuralModel
from evaluation.metrics import compute_metrics
from attacks.capgd import capgd_attack

rows = []

for seed in SEEDS:
    print(f"\n{'='*60}\n  SEED = {seed}\n{'='*60}")

    dataset = load_dataset(DATASET, config={"sample_frac": SAMPLE_FRAC})
    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
        dataset, test_size=0.2, val_size=0.2, random_state=seed,
    )
    preprocessor = DataPreprocessor(dataset.feature_types)
    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p  = preprocessor.transform(X_test)
    proc_ft = {c: "numeric" for c in X_train_p.columns}
    schema = ConstraintSchema.from_data(X_train_p, proc_ft)

    scaler, num_names = get_scaler_and_num_names(preprocessor)
    ohe_blocks = build_ieee_ohe_blocks(X_train_p.columns.tolist())
    print(f"  proc_dim={X_test_p.shape[1]}  ohe_blocks={[(b['prefix'], len(b['indices'])) for b in ohe_blocks]}")

    model = NeuralModel(MODEL_PARAMS)
    t0 = time.time()
    model.fit(X_train_p, y_train)
    print(f"  trained in {time.time()-t0:.1f}s")
    clean_probs = model.predict_proba(X_test_p)
    clean_m = compute_metrics(y_test, clean_probs)
    print(f"  clean  PR-AUC={clean_m['pr_auc']:.4f}  Acc={clean_m['accuracy']:.4f}  "
          f"Rec={clean_m.get('recall', float('nan')):.4f}")

    X_test_raw = inverse_transform_numeric(X_test_p, num_names, scaler)
    _, clean_agg = ieee_feasibility(X_test_raw, X_test_p)

    # Unconstrained CAPGD
    t0 = time.time()
    X_adv_u = capgd_attack(model, X_test_p, y_test, schema, proc_ft, params=ATTACK_PARAMS)
    dt_u = time.time() - t0
    probs_u = model.predict_proba(X_adv_u)
    m_u = compute_metrics(y_test, probs_u)
    X_adv_u_raw = inverse_transform_numeric(X_adv_u, num_names, scaler)
    per_u, agg_u = ieee_feasibility(X_adv_u_raw, X_adv_u)

    # OHE-projected CAPGD
    t0 = time.time()
    X_adv_p = capgd_attack_ohe_projected(
        model, X_test_p, y_test, schema, proc_ft,
        ohe_blocks=ohe_blocks, params=ATTACK_PARAMS,
    )
    dt_p = time.time() - t0
    probs_p = model.predict_proba(X_adv_p)
    m_p = compute_metrics(y_test, probs_p)
    X_adv_p_raw = inverse_transform_numeric(X_adv_p, num_names, scaler)
    per_p, agg_p = ieee_feasibility(X_adv_p_raw, X_adv_p)

    parq_u = os.path.join(OUT_DIR, f"{DATASET}_neural_unconstrained_seed{seed}.parquet")
    parq_p = os.path.join(OUT_DIR, f"{DATASET}_neural_oheproj_seed{seed}.parquet")
    X_adv_u.to_parquet(parq_u)
    X_adv_p.to_parquet(parq_p)

    clean_pred = (clean_probs >= 0.5).astype(int)
    u_pred     = (probs_u    >= 0.5).astype(int)
    p_pred     = (probs_p    >= 0.5).astype(int)
    pos_mask = (y_test.values == 1)
    flipped_u = int(((clean_pred == 1) & (u_pred == 0) & pos_mask).sum())
    flipped_p = int(((clean_pred == 1) & (p_pred == 0) & pos_mask).sum())

    def _agg_mask(X_raw, X_p):
        idx = X_p.index
        parts = [
            _bool_series(check_txn_amt_positive(X_raw),         idx),
            _bool_series(check_c_nonneg(X_raw),                 idx),
            _bool_series(check_d_nonneg(X_raw),                 idx),
            _bool_series(check_ohe_valid(X_p, "ProductCD"),     idx),
            _bool_series(check_ohe_valid(X_p, "card4"),         idx),
            _bool_series(check_ohe_valid(X_p, "card6"),         idx),
        ]
        agg = parts[0]
        for q in parts[1:]:
            agg = agg & q
        return agg.values

    mask_u = _agg_mask(X_adv_u_raw, X_adv_u)
    mask_p = _agg_mask(X_adv_p_raw, X_adv_p)
    flipped_feas_u = int(((clean_pred == 1) & (u_pred == 0) & pos_mask & mask_u).sum())
    flipped_feas_p = int(((clean_pred == 1) & (p_pred == 0) & pos_mask & mask_p).sum())

    print(f"  unconst  PR-AUC={m_u['pr_auc']:.4f}  Acc={m_u['accuracy']:.4f}  "
          f"agg_feas={agg_u:.4f}  prod={per_u['i_product_ohe']:.3f} card4={per_u['i_card4_ohe']:.3f} card6={per_u['i_card6_ohe']:.3f}  "
          f"flipped={flipped_u} feas-flip={flipped_feas_u}  ({dt_u:.1f}s)")
    print(f"  oheproj  PR-AUC={m_p['pr_auc']:.4f}  Acc={m_p['accuracy']:.4f}  "
          f"agg_feas={agg_p:.4f}  prod={per_p['i_product_ohe']:.3f} card4={per_p['i_card4_ohe']:.3f} card6={per_p['i_card6_ohe']:.3f}  "
          f"flipped={flipped_p} feas-flip={flipped_feas_p}  ({dt_p:.1f}s)")

    def _row(attack, metrics, per, agg, flipped, flipped_feas, dt):
        r = {
            "seed": seed, "attack": attack,
            "clean_pr_auc": clean_m["pr_auc"], "clean_accuracy": clean_m["accuracy"],
            "clean_recall": clean_m.get("recall", np.nan),
            "robust_pr_auc": metrics["pr_auc"], "robust_accuracy": metrics["accuracy"],
            "robust_recall": metrics.get("recall", np.nan),
            "clean_feasibility": clean_agg,
            "adv_feasibility":   agg,
            "flipped_positives": flipped,
            "feasible_flipped":  flipped_feas,
            "attack_time_sec":   round(dt, 1),
        }
        for k, v in per.items(): r[f"adv_{k}"] = v
        return r

    rows.append(_row("unconstrained", m_u, per_u, agg_u, flipped_u, flipped_feas_u, dt_u))
    rows.append(_row("oheproj",       m_p, per_p, agg_p, flipped_p, flipped_feas_p, dt_p))

results_df = pd.DataFrame(rows)
results_csv = os.path.join(OUT_DIR, "ieee_ohe_projection_results.csv")
results_df.to_csv(results_csv, index=False)
print(f"\nSaved {len(results_df)} rows -> {results_csv}")
results_df


In [ ]:
# Cell 11: Summary - mean +/- std across seeds, OHE-projected vs unconstrained
agg = results_df.groupby("attack").agg(
    robust_pr_auc_mean   =("robust_pr_auc",    "mean"),
    robust_pr_auc_std    =("robust_pr_auc",    "std"),
    robust_accuracy_mean =("robust_accuracy",  "mean"),
    robust_accuracy_std  =("robust_accuracy",  "std"),
    adv_feasibility_mean =("adv_feasibility",  "mean"),
    adv_feasibility_std  =("adv_feasibility",  "std"),
    adv_product_mean     =("adv_i_product_ohe", "mean"),
    adv_card4_mean       =("adv_i_card4_ohe",  "mean"),
    adv_card6_mean       =("adv_i_card6_ohe",  "mean"),
    adv_d_nonneg_mean    =("adv_i_d_nonneg",   "mean"),
    flipped_positives_mean=("flipped_positives","mean"),
    feasible_flipped_mean=("feasible_flipped", "mean"),
).reset_index()
agg["filtered_success_rate"] = agg["feasible_flipped_mean"] / agg["flipped_positives_mean"]

summary_csv = os.path.join(OUT_DIR, "ieee_ohe_projection_summary.csv")
agg.to_csv(summary_csv, index=False)

print("Two attack regimes on IEEE-CIS, mean +/- std over 3 seeds:")
print(agg.to_string(index=False))
print(f"\nSaved -> {summary_csv}")


In [ ]:
# Cell 12: Drive backup + comparison plot
import shutil
import matplotlib.pyplot as plt

DRIVE_OUT = "/content/drive/MyDrive/FraudBench/results/adv_examples/ieee_ohe_projection"
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in sorted(os.listdir(OUT_DIR)):
    shutil.copy(os.path.join(OUT_DIR, f), os.path.join(DRIVE_OUT, f))
print(f"Backed up {len(os.listdir(OUT_DIR))} files -> {DRIVE_OUT}")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
attacks = agg["attack"].tolist()
x = np.arange(len(attacks))

axes[0].bar(x, agg["adv_feasibility_mean"], yerr=agg["adv_feasibility_std"].fillna(0), capsize=4)
axes[0].set_xticks(x); axes[0].set_xticklabels(attacks)
axes[0].set_ylabel("Aggregate adv feasibility")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Adv feasibility")

axes[1].bar(x, agg["filtered_success_rate"])
axes[1].set_xticks(x); axes[1].set_xticklabels(attacks)
axes[1].set_ylabel("feas-flipped / flipped")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Filtered success rate")

axes[2].bar(x, agg["flipped_positives_mean"])
axes[2].set_xticks(x); axes[2].set_xticklabels(attacks)
axes[2].set_ylabel("Flipped positives (count)")
axes[2].set_title("Flipped positives")

axes[3].bar(x, agg["robust_pr_auc_mean"], yerr=agg["robust_pr_auc_std"].fillna(0), capsize=4)
axes[3].set_xticks(x); axes[3].set_xticklabels(attacks)
axes[3].set_ylabel("Robust PR-AUC")
axes[3].set_title("Robust PR-AUC (model damage)")

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "ieee_ohe_projection_bars.png")
plt.savefig(fig_path, dpi=150)
shutil.copy(fig_path, os.path.join(DRIVE_OUT, "ieee_ohe_projection_bars.png"))
plt.show()
print(f"Saved comparison plot -> {fig_path}")
